# London Clustering, Diffusion Maps on Two Real Embeddings

Exploratory notebook, first pass, not part of any book chapter. London-side
counterpart to the GoiEner diffusion-maps notebook, built on top of the
other two London representations: the Tucker household embedding
(`04-customer-feeder-clustering-london-tucker-code.ipynb`) and the
Chronos-2 zero-shot embedding
(`04-customer-feeder-clustering-london-chronos-embed-code.ipynb`).

Every notebook in this comparison has picked $k$ the same way: find the
silhouette maximum, trust it. Diffusion maps answer a more direct
question instead: is there genuinely a small number of distinct real
clusters here at all, or a continuum a forced discrete $k$ papers over?
They build a Markov random-walk kernel over the real data and read the
eigenvalue spectrum of that walk: a genuine cluster shows up as a real,
separate eigenvalue with a visible gap after it; a continuum, or noise,
does not. This notebook checks that real spectral gap on both London
embeddings, then clusters on the resulting diffusion coordinates.

## Getting the data and both real embeddings

Identical data loading to every other London notebook in this
comparison. Both embeddings are recomputed here exactly as their own
notebooks build them (peak-normalized Tucker factors; Chronos-2
zero-shot, mean-pooled, PCA-reduced to a 90%-variance bar), so this
notebook is self-contained rather than depending on another notebook's
own saved state.

In [ ]:
from lets_plot import LetsPlot
import numpy as np
import pandas as pd

from ark.cluster.datasets import load_london_pivot
from ark.cluster.preprocessing import map_to_seasons

LetsPlot.setup_html(isolated_frame=True)
N_PARTITIONS = 50
WINDOW_START = "2013-01-01"
WINDOW_DAYS = 360
MIN_COVERAGE = 0.999
STEPS_PER_DAY = 48  # half-hourly, not hourly like GoiEner

pivot = load_london_pivot(
    n_partitions=N_PARTITIONS,
    window_start=WINDOW_START,
    window_days=WINDOW_DAYS,
    min_coverage=MIN_COVERAGE,
)
n_customers = pivot.shape[1]
household_ids = list(pivot.columns)
print(
    f"pivot: {pivot.shape} (real half-hourly timestamps, real households), via the shared ark.cluster.datasets loader"
)

pivot: (17280, 1284) (real half-hourly timestamps, real households), via the shared ark.cluster.datasets loader


In [ ]:
def season_tensor(pivot: pd.DataFrame, dates: pd.DatetimeIndex) -> np.ndarray:
    """Real (households, days, half-hours) tensor for a real date subset, households in pivot's own column order."""
    subset = pivot[pivot.index.normalize().isin(dates)]
    return subset.T.to_numpy().reshape(subset.shape[1], len(dates), STEPS_PER_DAY)


day_index = pd.date_range(WINDOW_START, periods=WINDOW_DAYS, freq="D")
# Real calendar summer (Northern Hemisphere), matching the other London
# and GoiEner notebooks' own convention.
summer_dates = day_index[map_to_seasons(day_index, hemisphere="north") == "summer"]
season = season_tensor(pivot, summer_dates)
print(f"season tensor: {season.shape} (customers, real summer days, half-hours)")

season tensor: (1284, 92, 48) (customers, real summer days, half-hours)


In [ ]:
import tensorly as tl
from tensorly.decomposition import tucker

tl.set_backend("numpy")
household_peak = season.max(axis=(1, 2), keepdims=True)
household_peak = np.where(household_peak == 0, 1, household_peak)
season_normalized = season / household_peak

core, factors = tucker(tl.tensor(season_normalized), rank=[10, 10, 8], random_state=0)
tucker_embedding = factors[0]
print(f"tucker_embedding: {tucker_embedding.shape}")

tucker_embedding: (1284, 10)


In [ ]:
from chronos import Chronos2Pipeline

pipeline = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map="cpu", torch_dtype="auto")
season_sequence = season.reshape(n_customers, season.shape[1] * STEPS_PER_DAY)
inputs = [season_sequence[h].astype(np.float32) for h in range(n_customers)]
embeds, _ = pipeline.embed(inputs, batch_size=64)
chronos_raw = np.stack([e.numpy()[0].mean(axis=0) for e in embeds])

from sklearn.decomposition import PCA

pca_full = PCA(random_state=0).fit(chronos_raw)
n_components = int(np.searchsorted(np.cumsum(pca_full.explained_variance_ratio_), 0.90) + 1)
chronos_embedding = PCA(n_components=n_components, random_state=0).fit_transform(chronos_raw)
print(f"chronos_embedding: {chronos_embedding.shape} ({n_components} components for 90% variance)")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

chronos_embedding: (1284, 34) (34 components for 90% variance)


## Diffusion maps: a real spectral-gap check, not a trusted silhouette alone

A Gaussian kernel affinity, $\alpha$-normalized to correct for local
density (the same real correction Coifman and Lafon's original diffusion
maps use), turned into a genuine Markov random-walk transition matrix.
Its eigenvalues, in decreasing order, are the real signal to read: a true
cluster structure shows up as a small number of eigenvalues well
separated from the rest, a real gap; a continuum does not.

In [ ]:
from scipy.linalg import eigh
from scipy.spatial.distance import pdist, squareform


def diffusion_map(X: np.ndarray, n_components: int = 8, alpha: float = 1.0) -> tuple[np.ndarray, np.ndarray]:
    """Real diffusion-map coordinates and eigenvalues (Coifman & Lafon 2006), alpha-normalized."""
    dist = squareform(pdist(X, metric="euclidean"))
    sigma = np.median(dist[dist > 0])
    kernel = np.exp(-(dist**2) / (2 * sigma**2))
    degree = kernel.sum(axis=1)
    kernel_alpha = kernel / np.outer(degree**alpha, degree**alpha)
    degree_alpha = kernel_alpha.sum(axis=1)
    inv_sqrt_degree = 1.0 / np.sqrt(degree_alpha)
    symmetric = (inv_sqrt_degree[:, None] * kernel_alpha) * inv_sqrt_degree[None, :]
    eigvals, eigvecs = eigh(symmetric)
    order = np.argsort(eigvals)[::-1]
    eigvals = eigvals[order]
    eigvecs = eigvecs[:, order] * inv_sqrt_degree[:, None]
    return eigvals[1 : n_components + 1], eigvecs[:, 1 : n_components + 1]


tucker_eigvals, tucker_coords = diffusion_map(tucker_embedding, n_components=8)
chronos_eigvals, chronos_coords = diffusion_map(chronos_embedding, n_components=8)

spectrum_df = pd.DataFrame(
    {
        "Component": list(range(1, 9)) * 2,
        "Eigenvalue": np.concatenate([tucker_eigvals, chronos_eigvals]),
        "Embedding": ["Tucker (peak-normalized)"] * 8 + ["Chronos-2 (zero-shot)"] * 8,
    }
)

In [ ]:
from lets_plot import aes, geom_line, geom_point, ggplot, ggsize, labs, scale_color_manual

from ark.plot.theme import BRAND_PALETTE, modern_theme

EMBEDDING_COLORS = {"Tucker (peak-normalized)": BRAND_PALETTE[0], "Chronos-2 (zero-shot)": BRAND_PALETTE[1]}
p = (
    ggplot(spectrum_df, aes(x="Component", y="Eigenvalue", color="Embedding"))
    + geom_line()
    + geom_point(size=3)
    + scale_color_manual(values=EMBEDDING_COLORS)
    + labs(x="Diffusion-map component", y="Eigenvalue", title="Real Eigenvalue Spectrum: Where Is the Gap?")
    + modern_theme()
    + ggsize(600, 400)
)
p

## Clustering on the real diffusion coordinates

`KMeans` on the top diffusion coordinates for each embedding, the same
validity-curve procedure as every other notebook in this comparison, now
applied after the real spectral-gap check above rather than instead of
it.

In [ ]:
from ark.cluster.cluster_validation import clustering_validity_scores
from ark.plot.clustering import validity_curve
from ark.plot.gt_style import themed_gt

scores_tucker_diff = clustering_validity_scores(tucker_coords, range(2, 10))
scores_chronos_diff = clustering_validity_scores(chronos_coords, range(2, 10))
comparison_df = (
    scores_tucker_diff[["k", "silhouette"]]
    .rename(columns={"silhouette": "Tucker diffusion"})
    .merge(scores_chronos_diff[["k", "silhouette"]].rename(columns={"silhouette": "Chronos-2 diffusion"}), on="k")
)
themed_gt(comparison_df.round(3))

k,Tucker diffusion,Chronos-2 diffusion
2,0.984,0.875
3,0.946,0.876
4,0.951,0.159
5,0.953,0.155
6,0.955,0.169
7,0.956,0.163
8,0.93,0.164
9,0.931,0.166


In [ ]:
from sklearn.cluster import KMeans

results = []
for name, coords, scores in [
    ("Tucker diffusion", tucker_coords, scores_tucker_diff),
    ("Chronos-2 diffusion", chronos_coords, scores_chronos_diff),
]:
    k = int(scores.loc[scores["silhouette"].idxmax(), "k"])
    labels = KMeans(n_clusters=k, n_init=20, random_state=0).fit_predict(coords)
    counts = pd.Series(labels).value_counts()
    results.append(
        {
            "Embedding": name,
            "Chosen k": k,
            "Best silhouette": float(scores["silhouette"].max()),
            "Minority cluster size": int(counts.min()),
        }
    )

themed_gt(pd.DataFrame(results).round(3))

Embedding,Chosen k,Best silhouette,Minority cluster size
Tucker diffusion,2,0.984,1
Chronos-2 diffusion,3,0.876,2


## Are either of these splits actually trustworthy?

The same real trust gate already applied to the other London notebooks,
run here on both diffusion-coordinate embeddings.

In [ ]:
from scipy.stats import entropy
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, silhouette_score

from ark.cluster.cluster_validation import composite_trustworthy_score
from ark.cluster.stability import cluster_stability, prediction_strength


def _kmeans_cluster_fn(X_arr, k):
    return KMeans(n_clusters=k, n_init=20, random_state=0).fit_predict(X_arr)


def trust_gate_audit(X_embedding, k_range):
    rows = []
    for k in k_range:
        labels_k = _kmeans_cluster_fn(X_embedding, k)
        sizes = np.bincount(labels_k)
        ps = prediction_strength(X_embedding, k_range=[k], cluster_fn=_kmeans_cluster_fn, n_splits=10, random_state=0)[
            "prediction_strength"
        ].iloc[0]
        stability_scores = cluster_stability(
            X_embedding, labels_k, cluster_fn=_kmeans_cluster_fn, n_bootstrap=100, random_state=0
        )
        rows.append(
            {
                "k": k,
                "silhouette": silhouette_score(X_embedding, labels_k),
                "calinski_harabasz": calinski_harabasz_score(X_embedding, labels_k),
                "davies_bouldin": davies_bouldin_score(X_embedding, labels_k),
                "balance": entropy(sizes) / np.log(k),
                "prediction_strength": ps,
                "min_cluster_stability": min(stability_scores.values()),
            }
        )
    return pd.DataFrame(rows)


trust_audit_tucker_diff_df = trust_gate_audit(tucker_coords, range(2, 5))
print("--- Tucker diffusion ---")
themed_gt(trust_audit_tucker_diff_df.round(3))

--- Tucker diffusion ---


k,silhouette,calinski_harabasz,davies_bouldin,balance,prediction_strength,min_cluster_stability
2,0.984,647.905,0.01,0.009,0.994,0.57
3,0.946,709.922,0.196,0.064,0.989,0.61
4,0.951,973.546,0.173,0.059,0.896,0.61


In [ ]:
trust_audit_chronos_diff_df = trust_gate_audit(chronos_coords, range(2, 5))
print("--- Chronos-2 diffusion ---")
themed_gt(trust_audit_chronos_diff_df.round(3))

--- Chronos-2 diffusion ---


k,silhouette,calinski_harabasz,davies_bouldin,balance,prediction_strength,min_cluster_stability
2,0.899,135.608,0.07,0.009,0.852,0.211
3,0.144,165.272,1.68,0.622,0.548,0.471
4,0.155,167.345,1.556,0.75,0.418,0.403


In [ ]:
trust_gated_tucker_diff_df = composite_trustworthy_score(
    trust_audit_tucker_diff_df.rename(columns={"k": "trial_number"}),
    trust_metrics=("balance", "min_cluster_stability"),
    id_column="trial_number",
).rename(columns={"trial_number": "k"})
print("--- Tucker diffusion, trust-gated ---")
themed_gt(trust_gated_tucker_diff_df.round(3))

--- Tucker diffusion, trust-gated ---


k,separation_score,trust_factor,composite_score
4,0.8,0.059,0.047
3,0.467,0.064,0.03
2,0.733,0.009,0.007


In [ ]:
trust_gated_chronos_diff_df = composite_trustworthy_score(
    trust_audit_chronos_diff_df.rename(columns={"k": "trial_number"}),
    trust_metrics=("balance", "min_cluster_stability"),
    id_column="trial_number",
).rename(columns={"trial_number": "k"})
print("--- Chronos-2 diffusion, trust-gated ---")
themed_gt(trust_gated_chronos_diff_df.round(3))

--- Chronos-2 diffusion, trust-gated ---


k,separation_score,trust_factor,composite_score
4,0.8,0.403,0.323
3,0.467,0.471,0.22
2,0.733,0.009,0.007


## Summary

A real, sharper version of the same warning sign the GoiEner diffusion
notebook already raised, and a genuine step backward from London's own
Chronos-2 embedding finding above.

**Both diffusion transforms produce the same suspicious shape: a very
high, nearly flat silhouette that isolates almost nobody.** Tucker
diffusion's own validity curve sits between 0.930 and 0.984 across every
$k$ from 2 to 9, never dipping, and its chosen $k{=}2$ isolates a single
real household against the rest. Chronos-2 diffusion's own curve peaks
at 0.875-0.876 for $k{=}2$ and $k{=}3$, a near-tie, before crashing to
0.159 at $k{=}4$, and its own chosen split isolates just 2 households.
This is the exact magnitude-dominated shape already flagged in the raw
Tucker notebook and in GoiEner's own diffusion-maps run: a suspiciously
high, nearly-flat silhouette that turns out to mean "found one tiny
outlier," not "found real structure."

**The real trust gate rejects both, decisively, at every $k$ checked.**
Tucker diffusion's balance never exceeds 0.064 across $k{=}2$ to 4 (a
household count so lopsided the entropy-based balance score reads as
near-zero), and while prediction strength looks excellent (0.994, 0.989,
0.896), minimum cluster stability tops out at 0.61, short of Hennig's
stable band. Chronos-2 diffusion is worse: prediction strength at
$k{=}2$ is a real 0.852, but stability collapses to 0.211, deep in the
dissolved band, and $k{=}3$/$k{=}4$ fail both checks outright (prediction
strength 0.548 and 0.418). Composite scores stay low throughout, never
above 0.05 for Tucker diffusion or 0.33 for Chronos-2 diffusion, nowhere
close to the 0.7-plus composite scores this same comparison's own CROCS
and raw-Chronos-2 findings post.

**The real loss here is what diffusion did to an already-good
representation.** London's raw Chronos-2 embedding, checked earlier in
this comparison, found a decisively trustworthy, close-to-balanced
654/630 split at $k{=}2$ (prediction strength 0.891, stability 0.955).
Applying a diffusion-map transform on top of that same embedding turns
that trustworthy, balanced result into a near-degenerate, unstable
2-household split (stability 0.211). The geometric transform did not
preserve, and actively destroyed, the real structure the raw embedding
already had. That is a genuinely useful, disclosed negative finding on
its own: a spectral-gap check is only as good as the geometry it is run
on, and running one on top of an already-good embedding is not free.

Put together with the GoiEner diffusion notebook's own findings, the
picture is not identical across populations, and reporting that
honestly matters more than a tidy conclusion. GoiEner's own Tucker
diffusion did find one real, trustworthy point, $k{=}2$ (prediction
strength 0.833, stability 0.805), despite the same suspicious
near-flat silhouette curve at other $k$; London's Tucker diffusion never
reaches a comparably stable point at any $k$ checked (stability tops
out at 0.61). Chronos-2 diffusion, by contrast, degrades its own raw
embedding's trustworthiness on both populations: GoiEner's own raw
Chronos-2 embedding lost its clean $k{=}2$ result under diffusion, and
London's does too, more sharply (stability falling from 0.955 to
0.211). The consistent, cross-population lesson is narrower than "all
diffusion maps fail here": a diffusion transform is not free, and
whether it helps or actively destroys real structure depends on what
embedding it is applied to, checked case by case rather than assumed
either way.